# Hybrid Mamba-xLSTM: Complete End-to-End Validation (✅ FIXED)

**Issue Fixed**: ✅ Real-time output shown | ✅ Checkpoints verified | ✅ Better error messages  
**Status**: Ready for Colab T4  
**Time**: ~2-2.5 hours 


## P1: GPU Check

In [ ]:
import subprocess, sys
print('=== GPU ==='); subprocess.run(['nvidia-smi'])
print(f'Python {sys.version}')

## P2: Clone & Install

In [ ]:
import os, subprocess
repo = '/content/hybrid_model_mamba_xlstm'
if not os.path.exists(repo):
    subprocess.run(['git', 'clone', '--branch', 'a100_70m_baseline',
        'https://github.com/krishankb-de/hybrid_model_mamba_xlstm.git', repo])
os.chdir(repo); print('✅ Repo cloned')

In [ ]:
import subprocess, os
os.chdir('/content/hybrid_model_mamba_xlstm')
print('Installing...')
subprocess.run(['pip', 'install', '-e', '.'], timeout=300)

## P3: Unit Tests

In [ ]:
import subprocess, os
os.chdir('/content/hybrid_model_mamba_xlstm')
result = subprocess.run(['pytest', 'tests/test_encoder_pooling.py', '-v'], timeout=60)
print('✅ Tests' if result.returncode==0 else '❌ Tests failed')

## P4: Stage 0 Vanilla - ⭐ FIXED (Real-time output shown below)

In [ ]:
import subprocess, os, glob
os.chdir('/content/hybrid_model_mamba_xlstm')
os.makedirs('outputs', exist_ok=True)

print("\n" + "="*60)
print("STAGE 0: VANILLA LM (500 steps)")
print("="*60 + "\n")

cmd = ['python', 'scripts/train.py',
    'model=hybrid_70m', 'dataset=pubmed', 'trainer=a100_single_gpu',
    'trainer.batch_size=8', 'trainer.max_steps=500',
    'trainer.default_root_dir=outputs/colab_stage0_vanilla_sample',
    'experiment_name=colab_stage0_vanilla_sample']

# KEY FIX: Run WITHOUT capture_output to see real-time output
result = subprocess.run(cmd, timeout=3600)

# Verify checkpoint
ckpt = 'outputs/colab_stage0_vanilla_sample/checkpoints/last.ckpt'
if os.path.exists(ckpt):
    size = os.path.getsize(ckpt) / (1024**2)
    print(f"\n✅ CHECKPOINT SAVED: {size:.1f} MB")
else:
    print(f"\n❌ CHECKPOINT NOT FOUND: {ckpt}")
    ckpts = glob.glob('outputs/**/last.ckpt', recursive=True)
    print(f"   Found at: {ckpts if ckpts else 'NONE'}")

## P5: Stage 1 Vanilla - ⭐ FIXED (Real-time output shown below)

In [ ]:
import subprocess, os
os.chdir('/content/hybrid_model_mamba_xlstm')

s0_ckpt = 'outputs/colab_stage0_vanilla_sample/checkpoints/last.ckpt'
if not os.path.exists(s0_ckpt):
    print(f"❌ Stage 0 checkpoint missing: {s0_ckpt}")
else:
    print("\n" + "="*60)
    print("STAGE 1: VANILLA SIMCSE (200 steps)")
    print("="*60 + "\n")
    
    cmd = ['python', 'scripts/train_contrastive.py',
        'model=hybrid_70m', 'dataset=pubmed', 'trainer=a100_single_gpu',
        'trainer.batch_size=16', 'trainer.max_steps=200',
        f'checkpoint={s0_ckpt}',
        'trainer.default_root_dir=outputs/colab_stage1_vanilla_sample',
        'experiment_name=colab_stage1_vanilla_sample']
    
    # KEY FIX: Real-time output
    result = subprocess.run(cmd, timeout=3600)
    
    # Verify
    ckpt = 'outputs/colab_stage1_vanilla_sample/checkpoints/last.ckpt'
    if os.path.exists(ckpt):
        size = os.path.getsize(ckpt) / (1024**2)
        print(f"\n✅ CHECKPOINT SAVED: {size:.1f} MB")
    else:
        print(f"\n❌ CHECKPOINT NOT FOUND: {ckpt}")

## P6: Validation - ⭐ FIXED (Better error messages)

In [ ]:
import os, json, glob
os.chdir('/content/hybrid_model_mamba_xlstm')

print("\n" + "="*60)
print("FINAL VALIDATION REPORT")
print("="*60 + "\n")

ckpts = {
    'Stage 0': 'outputs/colab_stage0_vanilla_sample/checkpoints/last.ckpt',
    'Stage 1': 'outputs/colab_stage1_vanilla_sample/checkpoints/last.ckpt',
}

report = {}
all_ok = True

for name, path in ckpts.items():
    if os.path.exists(path):
        size = os.path.getsize(path) / (1024**2)
        print(f"{name:15s} ✅ READY ({size:.1f} MB)")
        report[name] = {"status": "PASS", "size_mb": f"{size:.1f}"}
    else:
        print(f"{name:15s} ❌ MISSING")
        report[name] = {"status": "MISSING"}
        all_ok = False

print("\n" + "="*60)
if all_ok:
    print("✅ SUCCESS: All checkpoints ready for download!")
else:
    print("❌ Some checkpoints failed")
    print("\nDebug:")
    print("  • Check errors in Stage 0/1 output above")
    print("  • Verify Stage 0 completed before Stage 1")
    os.system("ls -la outputs/ 2>/dev/null || true")

with open('colab_validation_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print("\n✅ Report: colab_validation_report.json")